# UCI HAR Data Preprocessing

This notebook constructs a reproducible preprocessing pipeline for the raw inertial signals in the UCI Human Activity Recognition dataset.

The objectives are to:

1. load and combine the nine sensor channels;
2. convert the activity labels to zero-based class indices;
3. create a subject-based validation split;
4. standardise the sensor data without data leakage;
5. verify the processed arrays; and
6. save the processed data for subsequent modelling.

No model training is performed in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Locate the project root whether the notebook is launched
# from the project directory or the notebooks directory.
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data" / "UCI_HAR"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {DATA_DIR}")

Project root: D:\Lunacia\Machine Learning
Dataset directory: D:\Lunacia\Machine Learning\data\UCI_HAR


## Loading the Original Dataset Splits

The inertial signals are stored in nine separate text files for each dataset split.

Each file contains one sensor channel with the shape:

`(samples, time steps)`

The nine channels will be combined into a three-dimensional array with the shape:

`(samples, time steps, sensor channels)`

In [22]:
SENSOR_CHANNELS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]

In [4]:
def load_dataset_split(split_name):
    split_dir = DATA_DIR / split_name
    signal_dir = split_dir / "Inertial Signals"

    signals = []

    for channel in SENSOR_CHANNELS:
        file_path = signal_dir / f"{channel}_{split_name}.txt"

        signal = np.loadtxt(
            file_path,
            dtype=np.float32,
        )

        signals.append(signal)

    features = np.stack(
        signals,
        axis=-1,
    )

    labels = np.loadtxt(
        split_dir / f"y_{split_name}.txt",
        dtype=np.int64,
    )

    subjects = np.loadtxt(
        split_dir / f"subject_{split_name}.txt",
        dtype=np.int64,
    )

    return features, labels, subjects

In [5]:
X_train_original, y_train_original, subject_train_original = (
    load_dataset_split("train")
)

X_test_original, y_test_original, subject_test = (
    load_dataset_split("test")
)

In [6]:
print(f"Training features: {X_train_original.shape}")
print(f"Training labels: {y_train_original.shape}")
print(f"Training subjects: {subject_train_original.shape}")

print()

print(f"Test features: {X_test_original.shape}")
print(f"Test labels: {y_test_original.shape}")
print(f"Test subjects: {subject_test.shape}")

Training features: (7352, 128, 9)
Training labels: (7352,)
Training subjects: (7352,)

Test features: (2947, 128, 9)
Test labels: (2947,)
Test subjects: (2947,)


## Verifying the Loaded Data

Before preprocessing begins, the loaded arrays are checked to confirm that:

- the feature arrays contain 128 time steps and 9 sensor channels;
- the feature, label, and subject counts agree;
- all sensor values are finite; and
- the official training and test subjects do not overlap.

In [7]:
def verify_loaded_split(
    features,
    labels,
    subjects,
    split_name,
):
    number_of_samples = features.shape[0]

    assert features.ndim == 3
    assert features.shape[1] == 128
    assert features.shape[2] == len(SENSOR_CHANNELS)

    assert labels.ndim == 1
    assert subjects.ndim == 1

    assert labels.shape[0] == number_of_samples
    assert subjects.shape[0] == number_of_samples

    assert np.isfinite(features).all()

    print(f"{split_name} split passed all checks.")

In [8]:
verify_loaded_split(
    X_train_original,
    y_train_original,
    subject_train_original,
    "Training",
)

verify_loaded_split(
    X_test_original,
    y_test_original,
    subject_test,
    "Test",
)

Training split passed all checks.
Test split passed all checks.


In [10]:
original_split_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "samples": [
            X_train_original.shape[0],
            X_test_original.shape[0],
        ],
        "time_steps": [
            X_train_original.shape[1],
            X_test_original.shape[1],
        ],
        "channels": [
            X_train_original.shape[2],
            X_test_original.shape[2],
        ],
        "subjects": [
            len(training_subjects),
            len(test_subjects),
        ],
    }
)

original_split_summary

,split,samples,time_steps,channels,subjects
0,train,7352,128,9,21
1,test,2947,128,9,9


## Converting the Activity Labels

The original labels range from 1 to 6. They are converted to zero-based class indices from 0 to 5 for model training.

In [18]:
activity_labels = pd.read_csv(
    DATA_DIR / "activity_labels.txt",
    sep=r"\s+",
    names=["original_label", "activity"],
)

activity_labels["class_index"] = (
    activity_labels["original_label"] - 1
)

y_train_zero_based = y_train_original - 1
y_test = y_test_original - 1

assert np.array_equal(
    np.unique(y_train_zero_based),
    np.arange(6),
)

assert np.array_equal(
    np.unique(y_test),
    np.arange(6),
)

activity_labels

,original_label,activity,class_index
0,1,WALKING,0
1,2,WALKING_UPSTAIRS,1
2,3,WALKING_DOWNSTAIRS,2
3,4,SITTING,3
4,5,STANDING,4
5,6,LAYING,5


## Creating a Subject-Based Validation Split

The original training data are divided into training and validation sets by subject. This prevents sensor windows from the same participant appearing in both sets.

In [19]:
RANDOM_SEED = 42
VALIDATION_SUBJECT_FRACTION = 0.20

all_training_subjects = np.unique(
    subject_train_original
)

number_of_validation_subjects = round(
    len(all_training_subjects)
    * VALIDATION_SUBJECT_FRACTION
)

random_generator = np.random.default_rng(
    RANDOM_SEED
)

validation_subjects = np.sort(
    random_generator.choice(
        all_training_subjects,
        size=number_of_validation_subjects,
        replace=False,
    )
)

validation_mask = np.isin(
    subject_train_original,
    validation_subjects,
)

training_mask = ~validation_mask

In [20]:
X_train = X_train_original[training_mask]
y_train = y_train_zero_based[training_mask]
subject_train = subject_train_original[training_mask]

X_validation = X_train_original[validation_mask]
y_validation = y_train_zero_based[validation_mask]
subject_validation = subject_train_original[validation_mask]

X_test = X_test_original.copy()

In [21]:
assert set(subject_train).isdisjoint(
    set(subject_validation)
)

print(
    f"Training: {X_train.shape}, "
    f"{len(np.unique(subject_train))} subjects"
)

print(
    f"Validation: {X_validation.shape}, "
    f"{len(np.unique(subject_validation))} subjects"
)

print(
    f"Test: {X_test.shape}, "
    f"{len(np.unique(subject_test))} subjects"
)

print(
    "Validation subjects:",
    validation_subjects,
)

Training: (5952, 128, 9), 17 subjects
Validation: (1400, 128, 9), 4 subjects
Test: (2947, 128, 9), 9 subjects
Validation subjects: [ 3 16 22 23]


## Checking Class Coverage

The training, validation, and test sets are checked to ensure that all six activity classes remain represented after the subject-based split.

In [23]:
class_counts = pd.DataFrame(
    {
        "activity": activity_labels["activity"],
        "train": np.bincount(
            y_train,
            minlength=6,
        ),
        "validation": np.bincount(
            y_validation,
            minlength=6,
        ),
        "test": np.bincount(
            y_test,
            minlength=6,
        ),
    }
)

assert (
    class_counts[
        ["train", "validation", "test"]
    ] > 0
).all().all()

class_counts

,activity,train,validation,test
0,WALKING,1012,214,496
1,WALKING_UPSTAIRS,870,203,471
2,WALKING_DOWNSTAIRS,800,186,420
3,SITTING,1035,251,491
4,STANDING,1104,270,532
5,LAYING,1131,276,537


## Standardising and Saving the Processed Data

Each sensor channel is standardised using the mean and standard deviation calculated from the training set only. The processed arrays and preprocessing metadata are then saved for subsequent modelling.

In [25]:
channel_mean = X_train.mean(
    axis=(0, 1),
    keepdims=True,
    dtype=np.float64,
)

channel_std = X_train.std(
    axis=(0, 1),
    keepdims=True,
    dtype=np.float64,
)

assert np.all(channel_std > 0)

In [26]:
def standardise(features):
    return (
        (features - channel_mean) / channel_std
    ).astype(np.float32)


X_train = standardise(X_train)
X_validation = standardise(X_validation)
X_test = standardise(X_test)

In [28]:
standardisation_summary = pd.DataFrame(
    {
        "channel": SENSOR_CHANNELS,
        "mean": X_train.mean(
            axis=(0, 1),
            dtype=np.float64,
        ),
        "standard_deviation": X_train.std(
            axis=(0, 1),
            dtype=np.float64,
        ),
    }
)

standardisation_summary

,channel,mean,standard_deviation
0,body_acc_x,7.598662e-11,1.0
1,body_acc_y,3.875991e-12,1.0
2,body_acc_z,4.705301e-11,1.0
3,body_gyro_x,4.561322e-11,1.0
4,body_gyro_y,-1.035182e-11,1.0
5,body_gyro_z,-5.963648e-11,1.0
6,total_acc_x,-4.122399e-11,1.0
7,total_acc_y,-2.367725e-11,1.0
8,total_acc_z,-4.743931e-11,1.0


In [29]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

processed_data_path = (
    PROCESSED_DIR
    / "uci_har_preprocessed.npz"
)

In [32]:
np.savez_compressed(
    processed_data_path,

    X_train=X_train,
    y_train=y_train,
    subject_train=subject_train,

    X_validation=X_validation,
    y_validation=y_validation,
    subject_validation=subject_validation,

    X_test=X_test,
    y_test=y_test,
    subject_test=subject_test,

    channel_mean=channel_mean.reshape(-1),
    channel_std=channel_std.reshape(-1),

    sensor_channels=np.asarray(
        SENSOR_CHANNELS,
        dtype=np.str_,
    ),
    activity_names=np.asarray(
        activity_labels["activity"].tolist(),
        dtype=np.str_,
    ),

    validation_subjects=validation_subjects,
)

In [33]:
with np.load(
    processed_data_path,
    allow_pickle=False,
) as saved_data:
    print("Saved arrays:")

    for array_name in saved_data.files:
        array = saved_data[array_name]

        print(
            f"{array_name}: "
            f"shape={array.shape}, "
            f"dtype={array.dtype}"
        )

print()
print(f"Saved to: {processed_data_path}")

Saved arrays:
X_train: shape=(5952, 128, 9), dtype=float32
y_train: shape=(5952,), dtype=int64
subject_train: shape=(5952,), dtype=int64
X_validation: shape=(1400, 128, 9), dtype=float32
y_validation: shape=(1400,), dtype=int64
subject_validation: shape=(1400,), dtype=int64
X_test: shape=(2947, 128, 9), dtype=float32
y_test: shape=(2947,), dtype=int64
subject_test: shape=(2947,), dtype=int64
channel_mean: shape=(9,), dtype=float64
channel_std: shape=(9,), dtype=float64
sensor_channels: shape=(9,), dtype=<U11
activity_names: shape=(6,), dtype=<U18
validation_subjects: shape=(4,), dtype=int64

Saved to: D:\Lunacia\Machine Learning\data\processed\uci_har_preprocessed.npz


## Summary

The UCI HAR inertial signals have been converted into model-ready training, validation, and test arrays.

The preprocessing pipeline:

- combines the nine sensor channels;
- converts the activity labels to zero-based class indices;
- creates a subject-based validation split;
- standardises each channel using training-set statistics only;
- verifies the integrity of the processed data; and
- saves the processed arrays and preprocessing metadata.

The processed dataset is now ready for model development.